# NIH Chest X-ray — Full Dataset EDA
**AI4ALL Ignite 2026 | Chest X-ray Abnormality Detection: Making It Fair for Everyone**
*Group: Tianxin Dong, Junaid, Belema Roberts, Krisha Rathod*

---

### What this notebook covers
1. Setup & data loading
2. Basic structure & missing values
3. Label distribution & class imbalance
4. Demographic analysis (age, sex, view position)
5. Age-correlated prevalence (bias source 1)
6. Patient-level leakage analysis (bias source 2)
7. Metadata data errors & cleaning (bias source 3)
8. Intersectional analysis (disease × age × sex)
9. Patient-level stratified 70/15/15 split
10. Data quality report summary

> **Dataset:** NIH Chest X-ray (`Data_Entry_2017.csv`) — 112,120 images, 30,805 patients, 15 disease labels.  
> **Reference sample:** `sample_labels.csv` (5,606 images) used for initial EDA — patterns confirmed consistent with full dataset.


## 0. Setup & Imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

# ── Edit this path to point to your local copy of the full dataset ──
DATA_PATH = 'Data_Entry_2017.csv'

print('Libraries loaded.')

## 1. Load Data & Basic Structure

In [ ]:
df = pd.read_csv(DATA_PATH)

# Drop empty trailing column if present
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Age in this file is stored as plain integer — no suffix parsing needed
# (Unlike sample_labels.csv which uses '058Y' format)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
print('Dtypes:')
print(df.dtypes)
print(f'\nUnique images:   {df["Image Index"].nunique():,}')
print(f'Unique patients: {df["Patient ID"].nunique():,}')
print(f'Images per patient (mean): {len(df)/df["Patient ID"].nunique():.2f}')

## 2. Missing Values

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('No missing values in core fields.')
else:
    print('Missing values:')
    print(missing)

## 3. Label Distribution & Class Imbalance

In [ ]:
# Parse multi-label column
LABELS = sorted({l for labs in df['Finding Labels'] for l in labs.split('|')})
for lab in LABELS:
    df[lab] = df['Finding Labels'].apply(lambda s: lab in s.split('|')).astype(int)

label_counts = df[LABELS].sum().sort_values(ascending=False)
print('Label counts:')
print(label_counts.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: all labels
colors = ['#888780' if lab == 'No Finding' else '#378ADD' for lab in label_counts.index]
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='none')
axes[0].set_title('Label counts — full dataset (n=112,120)', fontsize=12)
axes[0].set_ylabel('Image count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, v in zip(axes[0].patches, label_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                 f'{int(v):,}', ha='center', va='bottom', fontsize=7)

# Right: healthy vs diseased
df['has_disease'] = (df['Finding Labels'] != 'No Finding').astype(int)
binary = df['has_disease'].value_counts()
n_norm = int(binary.get(0,0)); n_dis = int(binary.get(1,0))
axes[1].bar(['No Finding\n(Healthy)', 'Has Disease'], [n_norm, n_dis],
            color=['#888780','#D85A30'], edgecolor='none')
axes[1].set_title('Healthy vs diseased split', fontsize=12)
axes[1].set_ylabel('Image count')
for i, v in enumerate([n_norm, n_dis]):
    axes[1].text(i, v+300, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('plot_01_label_distribution.png', bbox_inches='tight')
plt.show()
print(f'Imbalance ratio No Finding : Disease ≈ {n_norm/n_dis:.2f} : 1')

## 4. Demographic Analysis

In [ ]:
# ── Age ──
print('Age summary stats:')
print(df['Patient Age'].describe())
outlier_count = (df['Patient Age'] > 100).sum()
print(f'\nImplausible ages (>100 yrs): {outlier_count} records')

# Clean age
df['Age_clean'] = df['Patient Age'].where(df['Patient Age'] <= 100, np.nan)
age_clean = df['Age_clean'].dropna()
print(f'Clean age range: {age_clean.min():.0f} – {age_clean.max():.0f} yrs')
print(f'Clean mean age: {age_clean.mean():.1f} yrs')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age histogram
axes[0].hist(age_clean, bins=35, color='#378ADD', edgecolor='none')
axes[0].axvline(age_clean.mean(), color='#D85A30', linestyle='--', linewidth=1.5,
                label=f'Mean {age_clean.mean():.1f} yrs')
axes[0].set_title('Patient age distribution (cleaned)', fontsize=11)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)

# Gender
gender_counts = df['Patient Gender'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values,
            color=['#378ADD','#D85A30'], edgecolor='none')
axes[1].set_title('Gender split', fontsize=11)
for i, (_, v) in enumerate(gender_counts.items()):
    axes[1].text(i, v+300, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

# View position
view_counts = df['View Position'].value_counts()
axes[2].bar(view_counts.index, view_counts.values,
            color=['#0F6E56','#BA7517'], edgecolor='none')
axes[2].set_title('View position (PA=standard / AP=bedside)', fontsize=11)
for i, (_, v) in enumerate(view_counts.items()):
    axes[2].text(i, v+300, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('plot_02_demographics.png', bbox_inches='tight')
plt.show()

In [ ]:
# Age band breakdown
bins = [0, 18, 40, 60, 80, 500]
bin_labels = ['0-17', '18-39', '40-59', '60-79', '80+']
df['AgeBin'] = pd.cut(df['Age_clean'], bins=bins, labels=bin_labels, right=False)

bin_sizes = df['AgeBin'].value_counts().sort_index()
print('Images per age band:')
for band, n in bin_sizes.items():
    print(f'  {band}: {n:,} ({n/len(df)*100:.1f}%)')

## 5. Bias Source 1 — Age-correlated Prevalence

**Finding:** Several disease labels skew significantly older or younger than the overall mean age.  
A model trained without awareness of this could learn patient age as a shortcut rather than the radiographic signal.


In [ ]:
overall_mean = df['Age_clean'].mean()
rows = []
for lab in LABELS:
    sub = df[df[lab]==1]['Age_clean'].dropna()
    rows.append({'Label': lab, 'n': len(sub),
                 'MeanAge': sub.mean(), 'Diff': sub.mean() - overall_mean})
age_df = pd.DataFrame(rows).sort_values('Diff', ascending=False)
print(f'Overall mean age (cleaned): {overall_mean:.1f} yrs')
print()
print(age_df.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Mean age by label (horizontal bar)
colors = ['#D85A30' if d >= 0 else '#378ADD' for d in age_df['Diff']]
axes[0].barh(age_df['Label'], age_df['MeanAge'], color=colors, edgecolor='none')
axes[0].axvline(overall_mean, color='#888780', linestyle='--', linewidth=1.5,
                label=f'Overall mean ({overall_mean:.1f} yrs)')
axes[0].set_xlabel('Mean patient age (years)')
axes[0].set_title('Mean age by label vs overall mean', fontsize=12)
axes[0].legend(fontsize=9)

# Prevalence by age band for key labels
top_labels = ['No Finding','Infiltration','Effusion','Atelectasis',
              'Pneumothorax','Mass','Nodule','Cardiomegaly']
prev_table = df.groupby('AgeBin', observed=True)[top_labels].mean() * 100
palette = ['#888780','#D85A30','#378ADD','#0F6E56',
           '#993556','#BA7517','#534AB7','#638922']
for i, lab in enumerate(top_labels):
    axes[1].plot(bin_labels, prev_table[lab], marker='o',
                 label=lab, color=palette[i], linewidth=2, markersize=5)
axes[1].set_xlabel('Age band')
axes[1].set_ylabel('Prevalence (%)')
axes[1].set_title('Label prevalence by age band', fontsize=12)
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('plot_03_age_correlated_prevalence.png', bbox_inches='tight')
plt.show()

## 6. Bias Source 2 — Patient-level Leakage Risk

**Finding:** 84.4% of images belong to a patient who has multiple images in the dataset.  
Splitting by image (not patient ID) causes the model to see the same patient in train and test — inflating performance.


In [ ]:
img_per_patient = df['Patient ID'].value_counts()
multi = img_per_patient[img_per_patient > 1]

print(f'Total unique patients:          {df["Patient ID"].nunique():,}')
print(f'Total images:                   {len(df):,}')
print(f'Mean images per patient:        {img_per_patient.mean():.2f}')
print(f'Max images per patient:         {img_per_patient.max()}')
print()
print(f'Patients with >1 image:         {len(multi):,} / {len(img_per_patient):,}  ({len(multi)/len(img_per_patient)*100:.1f}%)')
print(f'Images from multi-img patients: {multi.sum():,} / {len(df):,}  ({multi.sum()/len(df)*100:.1f}%)')
print()
print('⚠️  Splitting by image rather than patient ID risks leakage for 84.4% of images.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution of images per patient
val_counts = img_per_patient.value_counts().sort_index().head(15)
axes[0].bar(val_counts.index.astype(str), val_counts.values,
            color='#993556', edgecolor='none')
axes[0].set_title('Images per patient (top 15 counts)', fontsize=11)
axes[0].set_xlabel('Number of images')
axes[0].set_ylabel('Number of patients')

# Leakage risk summary
categories = ['Single-image\npatients (safe)', 'Multi-image\npatients (leakage risk)']
values = [len(img_per_patient) - len(multi), len(multi)]
axes[1].bar(categories, values, color=['#378ADD','#D85A30'], edgecolor='none')
axes[1].set_title('Patient leakage risk', fontsize=11)
axes[1].set_ylabel('Number of patients')
for i, v in enumerate(values):
    axes[1].text(i, v+100, f'{v:,}\n({v/len(img_per_patient)*100:.1f}%)',
                 ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('plot_04_leakage_risk.png', bbox_inches='tight')
plt.show()

## 7. Bias Source 3 — Label Noise & Metadata Errors

**Finding:** Labels are NLP-mined from radiology reports (~10% error rate). Metadata also contains impossible age values.


In [ ]:
# Age outliers
outliers = df[df['Patient Age'] > 100][['Image Index','Patient ID','Patient Age',
                                         'Patient Gender','Finding Labels']]
print(f'Records with age > 100 years: {len(outliers)}')
print()
print(outliers.to_string(index=False))

In [ ]:
# Gender consistency
gc = df.groupby('Patient ID')['Patient Gender'].nunique()
print(f'Patients with inconsistent gender across records: {(gc > 1).sum()}')

# No Finding co-occurrence
bad_labels = df[df['Finding Labels'].str.contains('No Finding') &
                df['Finding Labels'].str.contains(r'\|')]
print(f'"No Finding" co-occurring with another label:     {len(bad_labels)}')

# Duplicates
print(f'Duplicate Image Index rows:                       {df["Image Index"].duplicated().sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Show age distribution highlighting outliers
ax.hist(df['Patient Age'].clip(upper=120), bins=50,
        color='#378ADD', edgecolor='none', alpha=0.7, label='All records')
outlier_ages = df[df['Patient Age'] > 100]['Patient Age'].clip(upper=120)
ax.axvline(100, color='#D85A30', linestyle='--', linewidth=1.5,
           label='100 yr threshold')
ax.set_title('Age distribution — outliers clipped to 120 for visibility', fontsize=11)
ax.set_xlabel('Patient age (years)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('plot_05_age_outliers.png', bbox_inches='tight')
plt.show()

print(f'\n16 records have ages 148-414 yrs → set to NaN, images retained.')
print(f'Action: Age_clean column created with outliers removed.')

## 8. Intersectional Analysis — Disease × Age × Sex

In [ ]:
# Disease rate by gender
by_gender = df.groupby('Patient Gender')['has_disease'].mean().round(3)
print('Disease rate by gender:')
print(by_gender)

# Disease rate by age group
df['age_group'] = pd.cut(df['Age_clean'],
                         bins=[0, 30, 50, 70, 100],
                         labels=['0-30','31-50','51-70','71-100'])
by_age = df.groupby('age_group', observed=True)['has_disease'].mean().round(3)
print('\nDisease rate by age group:')
print(by_age)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Disease rate by gender
by_gender.plot(kind='bar', ax=axes[0], color=['#378ADD','#D85A30'], edgecolor='none')
axes[0].set_title('Disease rate by gender', fontsize=11)
axes[0].set_ylabel('Rate (proportion)')
axes[0].set_xticklabels(by_gender.index, rotation=0)
for i, v in enumerate(by_gender.values):
    axes[0].text(i, v+0.003, f'{v:.3f}', ha='center', fontsize=10)

# Disease rate by age group
by_age.plot(kind='bar', ax=axes[1], color='#534AB7', edgecolor='none')
axes[1].set_title('Disease rate by age group', fontsize=11)
axes[1].set_ylabel('Rate (proportion)')
axes[1].set_xticklabels(by_age.index, rotation=0)

# Prevalence of top labels by sex
top5 = ['Effusion','Atelectasis','Infiltration','Mass','Pneumothorax']
sex_prev = df.groupby('Patient Gender')[top5].mean() * 100
x = np.arange(len(top5)); w = 0.35
axes[2].bar(x - w/2, sex_prev.loc['M'], w, label='Male',
            color='#378ADD', edgecolor='none')
axes[2].bar(x + w/2, sex_prev.loc['F'], w, label='Female',
            color='#D85A30', edgecolor='none')
axes[2].set_xticks(x)
axes[2].set_xticklabels(top5, rotation=30, ha='right', fontsize=9)
axes[2].set_title('Label prevalence by sex (%)', fontsize=11)
axes[2].set_ylabel('Prevalence (%)')
axes[2].legend()

plt.tight_layout()
plt.savefig('plot_06_intersectional.png', bbox_inches='tight')
plt.show()

## 9. Patient-level Stratified 70/15/15 Split

Split is performed **by patient ID**, not by image, to prevent leakage.  
Stratified by gender to maintain sex balance across train/val/test.


In [ ]:
patient_df = df.groupby('Patient ID').agg(
    n_images=('Image Index','count'),
    Gender=('Patient Gender','first')
).reset_index()

train_ids, temp_ids = train_test_split(
    patient_df['Patient ID'], test_size=0.30,
    random_state=42, stratify=patient_df['Gender']
)
temp_df = patient_df[patient_df['Patient ID'].isin(temp_ids)]
val_ids, test_ids = train_test_split(
    temp_df['Patient ID'], test_size=0.50,
    random_state=42, stratify=temp_df['Gender']
)

df['Split'] = 'train'
df.loc[df['Patient ID'].isin(val_ids),  'Split'] = 'val'
df.loc[df['Patient ID'].isin(test_ids), 'Split'] = 'test'

print('Split summary:')
print(f'{"Split":8} {"Images":>8} {"Patients":>10} {"% Images":>10}')
print('-' * 40)
for split in ['train','val','test']:
    sub = df[df['Split']==split]
    print(f'{split:8} {len(sub):>8,} {sub["Patient ID"].nunique():>10,} {len(sub)/len(df)*100:>9.1f}%')

In [ ]:
# Verify zero patient overlap
sets = {s: set(df[df['Split']==s]['Patient ID']) for s in ['train','val','test']}
print('Patient overlap checks:')
print(f'  train ∩ val:  {len(sets["train"] & sets["val"])}')
print(f'  train ∩ test: {len(sets["train"] & sets["test"])}')
print(f'  val   ∩ test: {len(sets["val"]   & sets["test"])}')
print()
if not any([sets['train']&sets['val'], sets['train']&sets['test'], sets['val']&sets['test']]):
    print('✓ Zero patient overlap — no leakage')

# Gender balance check
print('\nGender distribution per split (%):')
print(df.groupby('Split')['Patient Gender']
      .value_counts(normalize=True).mul(100).round(1)
      .rename('pct').reset_index()
      .pivot(index='Patient Gender', columns='Split', values='pct'))

# Save
df[['Image Index','Patient ID','Finding Labels','Patient Age',
    'Age_clean','Patient Gender','View Position','Split']].to_csv(
    'data_split.csv', index=False)
print('\nSaved: data_split.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Split image counts
split_counts = df['Split'].value_counts().reindex(['train','val','test'])
axes[0].bar(split_counts.index, split_counts.values,
            color=['#378ADD','#0F6E56','#D85A30'], edgecolor='none')
axes[0].set_title('Images per split', fontsize=11)
axes[0].set_ylabel('Image count')
for i, v in enumerate(split_counts.values):
    axes[0].text(i, v+500, f'{v:,}', ha='center', fontsize=10)

# Label prevalence train vs test (top 8)
top8 = label_counts.index[1:9].tolist()  # exclude No Finding
train_prev = df[df['Split']=='train'][top8].mean() * 100
test_prev  = df[df['Split']=='test'][top8].mean() * 100
x = np.arange(len(top8)); w = 0.35
axes[1].bar(x - w/2, train_prev, w, label='Train', color='#378ADD', edgecolor='none')
axes[1].bar(x + w/2, test_prev,  w, label='Test',  color='#D85A30', edgecolor='none')
axes[1].set_xticks(x)
axes[1].set_xticklabels(top8, rotation=35, ha='right', fontsize=8)
axes[1].set_title('Label prevalence: train vs test (%)', fontsize=11)
axes[1].set_ylabel('Prevalence (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_07_split_summary.png', bbox_inches='tight')
plt.show()

## 10. Data Quality Report Summary

In [ ]:
print('=' * 58)
print('  DATA QUALITY REPORT — NIH Chest X-ray Full Dataset')
print('=' * 58)
print(f'  Total images:           {len(df):>10,}')
print(f'  Unique patients:        {df["Patient ID"].nunique():>10,}')
print(f'  Disease labels:         {len(LABELS):>10}')
print(f'  Mean age (clean):       {df["Age_clean"].mean():>9.1f} yrs')
print()
print('── ISSUE 1: CLASS IMBALANCE ────────────────────────────')
print(f'  No Finding:  {label_counts["No Finding"]:>6,} ({label_counts["No Finding"]/len(df)*100:.1f}%)')
print(f'  Hernia (min):{label_counts["Hernia"]:>6,} ({label_counts["Hernia"]/len(df)*100:.1f}%)')
print(f'  Action: class weighting + oversampling during training')
print()
print('── ISSUE 2: AGE-CORRELATED PREVALENCE ─────────────────')
print(f'  Hernia mean age:    63.2 yrs (+16.3 above mean)')
print(f'  Pneumonia mean age: 44.9 yrs ( -2.0 below mean)')
print(f'  Action: age-stratified eval + train/test splits')
print()
print('── ISSUE 3: LABEL NOISE ────────────────────────────────')
print(f'  NLP-mined labels — est. ~10% error rate')
print(f'  Pneumothorax: captures treated cases (chest drain)')
print(f'  Action: label smoothing + confidence-weighted loss')
print()
print('── ISSUE 4: METADATA ERRORS ────────────────────────────')
print(f'  Age > 100 records:  {(df["Patient Age"]>100).sum():>4} → Age_clean set to NaN')
print(f'  Gender inconsistent:{(gc>1).sum():>4}  (none found)')
print(f'  No Finding + other: {len(bad_labels):>4}  (none found)')
print()
print('── ISSUE 5: PATIENT LEAKAGE ────────────────────────────')
multi2 = img_per_patient[img_per_patient > 1]
print(f'  Multi-image patients:  {len(multi2):,} / {len(img_per_patient):,} ({len(multi2)/len(img_per_patient)*100:.1f}%)')
print(f'  Images at risk:        {multi2.sum():,} / {len(df):,} ({multi2.sum()/len(df)*100:.1f}%)')
print(f'  Action: patient-level 70/15/15 split (verified ✓)')
print()
print('── SPLIT ───────────────────────────────────────────────')
for split in ['train','val','test']:
    sub = df[df['Split']==split]
    print(f'  {split:5}: {len(sub):>7,} images | {sub["Patient ID"].nunique():>6,} patients')
print('=' * 58)